### 生成MD格式的图函数
****

In [11]:
def generate_mermaid_code(langgraph): 
    """从 LangGraph 生成 Mermaid 代码，支持条件边（虚线）""" 
    graph_obj = langgraph.get_graph() 
    
    # 开始构建 Mermaid 代码 
    mermaid_lines = [ 
        "---", 
        "config:", 
        "  flowchart:", 
        "    curve: linear", 
        "---", 
        "graph TD" 
    ] 
    
    # 添加节点定义 
    for node_id, node in graph_obj.nodes.items(): 
        node_name = node.name if hasattr(node, 'name') else node_id 
        if node_id == "__start__": 
            mermaid_lines.append(f"    {node_id}([<{node_name}>]):::first") 
        elif node_id == "__end__": 
            mermaid_lines.append(f"    {node_id}([<{node_name}>]):::last") 
        else: 
            mermaid_lines.append(f"    {node_id}({node_name})") 
    
    # 确定条件路由节点
    # 通常条件路由边是从具有条件路由函数的节点发出的
    conditional_source_nodes = set()
    
    # 在您的例子中，'a' 节点有 route 函数，所以从 'a' 发出的边是条件边
    # 我们可以检查图的结构来确定
    for edge in graph_obj.edges:
        # 如果源节点不是 __start__ 并且有多个输出边，这通常意味着它是条件路由节点
        out_edges_count = sum(1 for e in graph_obj.edges if e.source == edge.source)
        if edge.source != '__start__' and edge.source != '__end__' and out_edges_count > 1:
            conditional_source_nodes.add(edge.source)
    
    # 添加边定义，根据是否为条件边选择实线或虚线
    for edge in graph_obj.edges:
        if edge.source in conditional_source_nodes:
            # 条件边使用虚线
            mermaid_lines.append(f"    {edge.source} -.-> {edge.target}")
        else:
            # 普通边使用实线
            mermaid_lines.append(f"    {edge.source} --> {edge.target}")
    
    # 添加样式定义 
    mermaid_lines.extend([ 
        "", 
        "classDef default fill:#4A90E2,color:#ffffff,stroke:#000000,stroke-width:2px,line-height:1.2", 
        "classDef first fill:#00AA00,color:#ffffff,stroke:#000000,stroke-width:2px,fill-opacity:0.8", 
        "classDef last fill:#FF4444,color:#ffffff,stroke:#000000,stroke-width:2px,fill-opacity:0.8" 
    ]) 
    
    return "\n".join(mermaid_lines)

In [12]:
import sys, os

# 切到项目根目录
sys.path.insert(0, r"E:\private_project\AI_application")
os.chdir(r"E:\private_project\AI_application")

from agent.react_agent import ReactAgent

# 初始化（会加载模型、构建图，可能需要几秒）
agent = ReactAgent()
print("Agent 初始化完成")

2026-05-28 21:50:22,983 - agent - INFO - react_agent.py:58 - [ReactAgent] 初始化多智能体 Supervisor 系统...
2026-05-28 21:50:22,985 - agent - INFO - prompt_loader.py:20 - [prompt_loader] 成功加载: e:\private_project\AI_application\prompts/main_prompt.txt
2026-05-28 21:50:22,985 - agent - INFO - prompt_loader.py:20 - [prompt_loader] 成功加载: e:\private_project\AI_application\prompts/diagnosis_prompt.txt
2026-05-28 21:50:22,986 - agent - INFO - prompt_loader.py:20 - [prompt_loader] 成功加载: e:\private_project\AI_application\prompts/main_prompt.txt
2026-05-28 21:50:22,987 - agent - INFO - prompt_loader.py:20 - [prompt_loader] 成功加载: e:\private_project\AI_application\prompts/data_prompt.txt
2026-05-28 21:50:22,988 - agent - INFO - prompt_loader.py:20 - [prompt_loader] 成功加载: e:\private_project\AI_application\prompts/main_prompt.txt
2026-05-28 21:50:22,989 - agent - INFO - react_agent.py:66 - [ReactAgent] 构建 Diagnosis 子Agent...
2026-05-28 21:50:22,994 - agent - INFO - react_agent.py:75 - [ReactAgent] 构建 Data 子A

Agent 初始化完成


In [13]:
from IPython.display import Markdown, display
import re

# 1. 用 LangGraph 生成基础 Mermaid，然后清洗
base = agent.graph.get_graph().draw_mermaid()

# 去掉 HTML 实体（VS Code 渲染器不认 &nbsp;）
base = base.replace("&nbsp;", " ")
base = re.sub(r'-\..*?\.->', lambda m: m.group().replace(' ', ''), base)
# 也清理多余的 classDef
base = re.sub(r'classDef default.*', '', base)
base = re.sub(r'classDef first.*', '', base)
base = re.sub(r'classDef last.*', '', base)

# 2. 鲜艳配色 + 加粗
STYLES = """
classDef default fill:#1a1a2e,color:#fff,stroke:#e94560,stroke-width:2px,font-weight:bold
classDef first fill:#00b894,color:#fff,stroke:#00b894,stroke-width:2px,font-weight:bold
classDef last fill:#e17055,color:#fff,stroke:#e17055,stroke-width:2px,font-weight:bold
classDef supervisor fill:#6c5ce7,color:#fff,stroke:#a29bfe,stroke-width:3px,font-weight:bold
classDef agent fill:#0984e3,color:#fff,stroke:#74b9ff,stroke-width:2px,font-weight:bold
classDef validator fill:#fdcb6e,color:#1a1a2e,stroke:#f39c12,stroke-width:2px,font-weight:bold
classDef error fill:#d63031,color:#fff,stroke:#ff7675,stroke-width:3px,font-weight:bold
classDef fallback fill:#636e72,color:#fff,stroke:#b2bec3,stroke-width:2px,font-weight:bold
class supervisor supervisor
class diagnosis_agent,data_agent,monitor_agent,general_agent agent
class result_validator validator
class error_handler error
class fallback fallback"""

mermaid = base + STYLES

display(Markdown(f"```mermaid\n{mermaid}\n```"))
print("🟣紫色=Supervisor 🔵蓝=子Agent 🟡黄=Validator 🔴红=ErrorHandler ⚫灰=Fallback")

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	diagnosis_agent(diagnosis_agent)
	data_agent(data_agent)
	monitor_agent(monitor_agent)
	general_agent(general_agent)
	result_validator(result_validator)
	error_handler(error_handler)
	fallback(fallback)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	data_agent --> result_validator;
	diagnosis_agent --> result_validator;
	error_handler -.-> fallback;
	error_handler -.retry.-> supervisor;
	general_agent --> result_validator;
	monitor_agent --> result_validator;
	result_validator -.ok.-> __end__;
	result_validator -.error.-> error_handler;
	supervisor -.-> data_agent;
	supervisor -.-> diagnosis_agent;
	supervisor -.-> fallback;
	supervisor -.-> general_agent;
	supervisor -.-> monitor_agent;
	fallback --> __end__;
	
	
	

classDef default fill:#1a1a2e,color:#fff,stroke:#e94560,stroke-width:2px,font-weight:bold
classDef first fill:#00b894,color:#fff,stroke:#00b894,stroke-width:2px,font-weight:bold
classDef last fill:#e17055,color:#fff,stroke:#e17055,stroke-width:2px,font-weight:bold
classDef supervisor fill:#6c5ce7,color:#fff,stroke:#a29bfe,stroke-width:3px,font-weight:bold
classDef agent fill:#0984e3,color:#fff,stroke:#74b9ff,stroke-width:2px,font-weight:bold
classDef validator fill:#fdcb6e,color:#1a1a2e,stroke:#f39c12,stroke-width:2px,font-weight:bold
classDef error fill:#d63031,color:#fff,stroke:#ff7675,stroke-width:3px,font-weight:bold
classDef fallback fill:#636e72,color:#fff,stroke:#b2bec3,stroke-width:2px,font-weight:bold
class supervisor supervisor
class diagnosis_agent,data_agent,monitor_agent,general_agent agent
class result_validator validator
class error_handler error
class fallback fallback
```

🟣紫色=Supervisor 🔵蓝=子Agent 🟡黄=Validator 🔴红=ErrorHandler ⚫灰=Fallback
